# SetUp

In [32]:
import os

# Si la ruta actual termina en 'notebooks', subimos dos niveles
if os.getcwd().endswith("notebooks"):
    os.chdir("../../")
    
print(f"Directorio de trabajo actual: {os.getcwd()}")

Directorio de trabajo actual: /home/alanh/Dev/owns/thesis


# Neat

In [33]:
import jax, jax.numpy as jnp
from tensorneat import algorithm, genome
from tensorneat.pipeline import Pipeline
from tensorneat.genome import DefaultGenome, BiasNode, DefaultMutation
from tensorneat.problem.func_fit import CustomFuncFit
from tensorneat.common import ACT, AGG, State
from tensorneat.common.functions import act_jnp
from tensorneat.pipeline import Pipeline
from tensorneat.algorithm.neat import NEAT
from tensorneat.algorithm.hyperneat import HyperNEAT, FullSubstrate

POPSIZE = 200
MUTATION_ROUNDS = 20
GRAD_STEPS = 1
GRAD_LR = 0.002
LOG_EVERY = 1
SEED = 42
MAX_NODES = 200
MAX_CONNS = 5000
state = State(randkey=jax.random.key(SEED))
N_EPOCHS = 3
PATIENCE = 10        # generaciones sin mejora antes de cambiar cristal
MIN_DELTA = 0.001    # mejora mínima que cuenta

DATA_PATH = "crystals/"
CIF_PATH = DATA_PATH + "cif/"
METADATA_PATH = DATA_PATH + "_metadata.json"
EMBEDDINGS_PATH = DATA_PATH + "_embeddings.json"
FINAL_DATA_PATH = DATA_PATH + "_data.json"
DATASET_PATH = DATA_PATH + "dataset_fase1a.h5" 

## Data

### utils

In [34]:
import h5py
import jax
import jax.numpy as jnp

def load_crystal_dataset(h5_path):
    """
    Carga el dataset HDF5 y lo inyecta directamente en la memoria de JAX.
    """
    print(f"📦 Cargando datos desde: {h5_path}")
    
    with h5py.File(h5_path, 'r') as hf:
        # 1. Leer los datos como arreglos de NumPy estándar
        np_inputs = hf['inputs'][:]
        np_targets = hf['targets'][:]
        
        # Opcional: leer los IDs si necesitas rastrear qué material es cuál
        # Usamos .astype(str) para convertir los bytes de HDF5 a strings legibles
        np_ids = hf['material_ids'][:].astype(str)

    # 2. El paso mágico: Convertir a JAX Arrays (DeviceArray)
    # Esto empuja los datos a la GPU (o CPU si JAX está configurado así)
    jax_inputs = jnp.array(np_inputs)
    jax_targets = jnp.array(np_targets)

    print("✅ Datos cargados en JAX con éxito.")
    print(f"📊 Shape Inputs:  {jax_inputs.shape}  (Muestras, Features)")
    print(f"📊 Shape Targets: {jax_targets.shape}  (Muestras, Output_Dim)")
    
    return jax_inputs, jax_targets, np_ids

### Load

In [35]:
X_train, Y_train, material_ids = load_crystal_dataset(DATASET_PATH)

# Comprobación de que JAX tiene el control
print(f"\n🔍 Dispositivo de almacenamiento: {X_train.devices()}")

# Ejemplo: Si quisieras aislar solo el Silicio (suponiendo que es el primer elemento)
silicio_input = X_train[0]
silicio_target = Y_train[0]

print(f"\n🎯 Evaluando objetivo: {material_ids[0]}")
print(f"   Lattice original (a, b, c, alpha, beta, gamma):")
print(f"   {silicio_target[:6]}")

INPUT_DIM = X_train.shape[1]  # Número de features por material
OUTPUT_DIM = Y_train.shape[1]  # Número de propiedades a predecir (

📦 Cargando datos desde: crystals/dataset_fase1a.h5
✅ Datos cargados en JAX con éxito.
📊 Shape Inputs:  (16, 85)  (Muestras, Features)
📊 Shape Targets: (16, 22)  (Muestras, Output_Dim)

🔍 Dispositivo de almacenamiento: {CudaDevice(id=0)}

🎯 Evaluando objetivo: mp-1550
   Lattice original (a, b, c, alpha, beta, gamma):
   [0.77399445 0.7739944  0.7739945  0.33333334 0.33333337 0.33333334]


## Custom: Problem/Activation Function

In [36]:
import jax.numpy as jnp

MAX_ATOMS = OUTPUT_DIM // 5

class CrystalLoss:
    """
    Encapsula el target actual y expone loss_fn compatible con g.grad().
    Actualiza el target llamando a .set_target(y) antes de cada cristal.
    """
    
    def __init__(self, max_atoms: int):
        self.max_atoms = max_atoms
        self._target = None  # se setea antes de cada cristal
    
    def set_target(self, y):
        """
        y: jnp.array de shape (1, OUTPUT_DIM)
        Llama a esto antes de cada cristal nuevo.
        """
        self._target = y
    
    def __call__(self, preds):
        """
        preds: shape (batch, OUTPUT_DIM) - predicciones de la red
        Retorna: scalar loss
        
        Compatible con la firma que espera g.grad():
            loss_fn: (outputs) -> scalar
        """
        assert self._target is not None, "Llama a set_target() antes de usar el loss"
        
        # Reshape a (batch, MAX_ATOMS, 5)
        pred_atoms   = preds.reshape(-1, self.max_atoms, 5)
        target_atoms = self._target.reshape(-1, self.max_atoms, 5)
        
        # Máscara: solo penalizar slots con átomo real
        mask = target_atoms[:, :, 0]  # (batch, MAX_ATOMS)
        
        # Loss 1: in_use - aprende cuántos átomos hay
        use_loss = jnp.mean(
            (pred_atoms[:, :, 0] - target_atoms[:, :, 0]) ** 2
        )
        
        # Loss 2: tipo de elemento - solo donde hay átomo real
        elem_loss = jnp.mean(
            mask * (pred_atoms[:, :, 1] - target_atoms[:, :, 1]) ** 2
        )
        
        # Loss 3: posiciones xyz - solo donde hay átomo real
        pos_loss = jnp.mean(
            mask[:, :, None] * (pred_atoms[:, :, 2:] - target_atoms[:, :, 2:]) ** 2
        )
        
        # Loss 4: penalización por coordenadas fuera de [0,1]
        # Las coordenadas fraccionarias deben estar en [0,1]
        coords = pred_atoms[:, :, 2:]  # (batch, MAX_ATOMS, 3)
        out_of_bounds = jnp.mean(
            mask[:, :, None] * jnp.maximum(0.0, coords - 1.0) ** 2 +
            mask[:, :, None] * jnp.maximum(0.0, -coords) ** 2
        )
        
        return (
            1.0  * use_loss    +
            2.0  * elem_loss   +
            10.0 * pos_loss    +
            5.0  * out_of_bounds
        )

## Algorithm

In [ ]:
neat = algorithm.NEAT(
    pop_size=POPSIZE,
    species_size=15,
    survival_threshold=0.01,
    genome=genome.DefaultGenome(
        num_inputs=INPUT_DIM,      # <-- Tu INPUT_DIM del Test Aislado
        num_outputs=OUTPUT_DIM,     # <-- Tu OUTPUT_DIM (Lattice + 4 Átomos)
        max_nodes=MAX_NODES,      # Permitimos cerebros más grandes
        max_conns=MAX_CONNS,      
        init_hidden_layers=(), # Dos capas ocultas iniciales para darle poder
        output_transform=act_jnp.identity_, # Salidas entre 0.0 y 1.0
        mutation=DefaultMutation(
            conn_add=0.6,
            conn_delete=0.3,
            node_add=0.1,
            node_delete=0.05,
        ),
    ),
)

## Pipeline

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

MAX_ATOMS = OUTPUT_DIM // 5
np_inputs  = np.array(X_train)
np_targets = np.array(Y_train)

state = neat.setup(state)
g = neat.genome
randkey = state.randkey

def make_loss_fn(y_target):
    def loss_fn(preds):
        pred_atoms   = preds.reshape(-1, MAX_ATOMS, 5)
        target_atoms = y_target.reshape(-1, MAX_ATOMS, 5)
        mask = target_atoms[:, :, 0]
        use_loss  = jnp.mean((pred_atoms[:, :, 0] - target_atoms[:, :, 0]) ** 2)
        elem_loss = jnp.mean(mask * (pred_atoms[:, :, 1] - target_atoms[:, :, 1]) ** 2)
        pos_loss  = jnp.mean(
            mask[:, :, None] * (pred_atoms[:, :, 2:] - target_atoms[:, :, 2:]) ** 2
        )
        coords = pred_atoms[:, :, 2:]
        oob = jnp.mean(
            mask[:, :, None] * (
                jnp.maximum(0.0, coords - 1.0) ** 2 +
                jnp.maximum(0.0, -coords) ** 2
            )
        )
        return use_loss + 2.0 * elem_loss + 10.0 * pos_loss + 5.0 * oob
    return loss_fn

N_GENERATIONS = 100
GRAD_STEPS_PER_GEN = 50  # steps de gradient descent por generación

historial = []

for generation in range(N_GENERATIONS):
    # 1. NEAT genera la población actual
    pop_nodes, pop_conns = neat.ask(state)

    # 2. Elegir UN cristal para esta generación
    idx = int(np.random.randint(len(material_ids)))
    x_current = jnp.array(np_inputs[idx:idx+1])
    y_current = jnp.array(np_targets[idx:idx+1])
    mid = material_ids[idx]

    # 3. Gradient descent sobre los pesos de toda la población
    loss_fn = make_loss_fn(y_current)

    def single_step(nodes, conns):
        loss, (grads_n, grads_c) = g.grad(
            state, nodes, conns, x_current, loss_fn
        )
        grads_n = jnp.clip(grads_n, -1.0, 1.0)
        grads_c = jnp.clip(grads_c, -1.0, 1.0)
        return nodes - GRAD_LR * grads_n, conns - GRAD_LR * grads_c, loss

    batch_grad = jax.jit(jax.vmap(single_step))

    for step in range(GRAD_STEPS_PER_GEN):
        pop_nodes, pop_conns, losses = batch_grad(pop_nodes, pop_conns)

    # 4. Fitness = negativo del loss (menor loss = mayor fitness)
    cpu_losses = jax.device_get(losses)
    valid = np.isfinite(cpu_losses)
    cpu_losses = np.where(valid, cpu_losses, 1e6)  # penalizar NaN
    fitnesses = -cpu_losses  # NEAT maximiza fitness

    # 5. NEAT selecciona, cruza y muta para la siguiente generación
    state = neat.tell(state, fitnesses)

    if generation % LOG_EVERY == 0:
        bl = float(np.min(cpu_losses[valid])) if valid.any() else float('nan')
        print(f"Gen {generation:3d} | cristal: {mid} | "
              f"best_loss: {bl:.6f} | válidos: {valid.sum()}/{POPSIZE}")

print("✅ Entrenamiento completado")

In [ ]:
# ── 4. Show result ───────────────────────────────────────────────────

cpu_losses = jax.device_get(losses)
best_idx = int(jnp.nanargmin(cpu_losses))
best = (pop_nodes[best_idx], pop_conns[best_idx])

print(f"\n🥇 Mejor individuo (loss global = {cpu_losses[best_idx]:.6f}):")
# Opcional: Imprime la estructura de la red (nodos y conexiones)
# print(g.repr(state, *best)) 

jit_batch_forward = jax.jit(jax.vmap(g.forward, in_axes=(None, None, 0)))
best_transformed = jax.jit(g.transform)(state, *best)

# Pasamos X_train en lugar del viejo prob.inputs
preds = jit_batch_forward(state, best_transformed, X_train)

print("\n📊 Resultados de Inferencia (Predicción vs Real):")
# Convertimos los tensores de JAX a Numpy estándar para poder imprimirlos bonito
cpu_preds = jax.device_get(preds)
cpu_targets = jax.device_get(Y_train)

for i in range(len(material_ids)):
    id_mat = material_ids[i]
    target_lat = cpu_targets[i][:6]
    pred_lat = cpu_preds[i][:6]
    
    print(f"\n🔹 {id_mat}:")
    print(f"   Lattice Real: {jnp.round(target_lat, 4)}")
    print(f"   Lattice IA  : {jnp.round(pred_lat, 4)}")


🥇 Mejor individuo (loss global = nan):

📊 Resultados de Inferencia (Predicción vs Real):

🔹 mp-1550:
   Lattice Real: [0.774  0.774  0.774  0.3333 0.3333 0.3333]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-2624:
   Lattice Real: [0.87469995 0.87469995 0.87469995 0.3333     0.3333     0.3333    ]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-1342:
   Lattice Real: [0.7892 0.7892 0.7892 0.3333 0.3333 0.3333]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-3347313:
   Lattice Real: [0.4913 0.4913 0.7391 0.4121 0.4121 0.3333]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-2469:
   Lattice Real: [0.83239996 0.83239996 0.83239996 0.3333     0.3333     0.3333    ]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-2691:
   Lattice Real: [0.8684 0.8684 0.8684 0.3333 0.3333 0.3333]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-406:
   Lattice Real: [0.92829996 0.92829996 0.92829996 0.3333     0.3333     0.3333    ]
   Lattice IA  : [nan nan nan nan nan nan]

🔹 mp-830:
   Lattice Re